# S08 - Machine-learning walk-forward on Welch-Goyal-style predictors (ML)

**Hypothesis.** No single predictor times the market reliably (Welch & Goyal 2008), but a
regularised model that combines valuation, rates, credit, trend and volatility might, if it is
trained strictly out of sample (Rapach, Strauss & Zhou 2010; Gu, Kelly & Xiu 2020).

**Model.** At each month-end *t*, predict P(next-month S&P 500 return > T-bill return) and hold
equities when it is > 0.5. Walk-forward: refit every 12 months on all earlier months (expanding,
first fit after 120 months), purge 1 month so no training label overlaps the test months, fresh
model per fold, fixed seed. Variants (fixed in advance): logistic regression C = 0.1 / 1
(standardised features), gradient boosting depth 2 / 3. Baseline: training-mean predictor
(`bt_ml.NaiveMean`), shown but not traded.

**Features and point-in-time lags**

| Feature | Source | Lag used |
|---|---|---|
| dividend yield D/P | Shiller monthly | month M usable from M+2 |
| earnings yield E/P | Shiller monthly (E interpolated from quarterly) | M+4 (extra quarter, interpolation) |
| term spread AAA - TB3MS, credit spread BAA - AAA, T-bill level | FRED monthly averages | M+1 + 7 days |
| 12-month and 1-month total return, 21-day realised vol, close / SMA200 - 1 | daily S&P 500 | up to the decision close |

Monthly check, next-close execution, 5 bp per switch. Shiller data end 2024-09, so the two
valuation features are frozen at their last value afterwards (holdout tail).

**Sources.** Welch & Goyal (2008), RFS doi:10.1093/rfs/hhm014; Goyal, Welch & Zafirov (2024),
SSRN 3929119; Rapach, Strauss & Zhou (2010), SSRN 1257858; Gu, Kelly & Xiu (2020), NBER w25398;
scikit-learn (BSD-3).

> Decision-support research only; never places orders.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
sys.path.insert(0, str(_root / "notebooks" / "backtest"))
import bt_common as bc  # noqa: E402
import bt_long  # noqa: E402
import bt_periods as bp  # noqa: E402
import bt_pit  # noqa: E402
import bt_registry as br  # noqa: E402
import bt_stats as bs  # noqa: E402
import seed_registry  # noqa: E402
import stage2 as s2  # noqa: E402

bc.setup_plotly("cdn")
pd.set_option("display.width", 180)
TRIAL_ID, STRATEGY, NOTEBOOK = 'S08', 'ml_walkforward', 'ml/ml_walkforward.ipynb'
PROTOCOL, REBALANCE = 'main', 'monthly'

sample = bt_long.long_sample()
px, cash = sample["px"], sample["cash"]
bench = bp.benchmark_returns(px, cash)  # SMA200 monthly / B&H / 60-40 / T-bills, frozen settings
print("prices", px.index[0].date(), "->", px.index[-1].date(), "| registry trials so far:", br.trial_count())

prices 1934-02-01 -> 2026-10-19 | registry trials so far: 23


## 1. Data and point-in-time inputs

In [2]:
import bt_ml

sh = sample["shiller"]
inputs = {"px": px, "cash": cash, "shiller": sh[["P", "D", "E"]].dropna()}
for s in ("BAA", "AAA", "TB3MS"):
    inputs[s] = bt_long.fred_series(s)
print("Shiller", sh.index[0], "->", sh.index[-1], "| FRED to", inputs["TB3MS"].index[-1].date())

Shiller 1871-01 -> 2024-09 | FRED to 2026-09-01


## 2. Signal and variant grid (fixed before any backtest)

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TRAIN_MIN, TEST_LEN, PURGE = 120, 12, 1  # months
FEATURES = ["dp", "ep", "term", "credit", "tbill", "mom12", "ret1", "vol21", "sma_dist"]


class Prob:
    # wraps a classifier so bt_ml.walk_forward_predict gets P(class 1) from .predict
    def __init__(self, est):
        self.est = est

    def fit(self, X, y):
        self.const = float(np.mean(y)) if len(np.unique(y)) < 2 else None
        if self.const is None:
            self.est.fit(X, y)
        return self

    def predict(self, X):
        return np.full(len(X), self.const) if self.const is not None else self.est.predict_proba(X)[:, 1]


def model_factory(kind, c=1.0, depth=2):
    if kind == "logit":
        return lambda: Prob(make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=2000)))
    if kind == "gbm":
        return lambda: Prob(HistGradientBoostingClassifier(max_depth=depth, max_iter=100, learning_rate=0.05, random_state=42))
    return bt_ml.NaiveMean


def features(inputs):
    px_, idx = inputs["px"], inputs["px"].index
    tr, close = px_["tr_close"], px_["close"]
    sh = inputs["shiller"].copy()
    sh.index = sh.index.to_timestamp(how="start")
    pit = bt_pit.monthly_to_daily_pit
    day = pd.DataFrame(index=idx)
    day["dp"] = pit(sh["D"] / sh["P"], idx, months=1)
    day["ep"] = pit(sh["E"] / sh["P"], idx, months=3)
    for name, ser in (("term", inputs["AAA"] - inputs["TB3MS"]), ("credit", inputs["BAA"] - inputs["AAA"]), ("tbill", inputs["TB3MS"])):
        day[name] = pit(ser.dropna(), idx, months=0, days=7)
    r = tr.pct_change(fill_method=None)
    day["mom12"] = tr / tr.shift(252) - 1
    day["ret1"] = tr / tr.shift(21) - 1
    day["vol21"] = r.rolling(21).std() * np.sqrt(252)
    day["sma_dist"] = close / close.rolling(200).mean() - 1
    dec = idx[bc.check_days(idx, "monthly").to_numpy()]
    X = day.loc[dec, FEATURES]
    ci = (1 + inputs["cash"].reindex(idx).fillna(0.0)).cumprod()
    nxt = pd.Series(dec, index=dec).shift(-1)
    ok = nxt.notna()
    y = pd.Series(np.nan, index=dec)
    y[ok] = (tr[nxt[ok]].to_numpy() / tr[dec[ok]].to_numpy() - 1) - (ci[nxt[ok]].to_numpy() / ci[dec[ok]].to_numpy() - 1)
    return X, (y > 0).astype(float).where(y.notna()), y


def predict(inputs, kind="logit", c=1.0, depth=2):
    X, ycls, _ = features(inputs)
    X = X.dropna()
    ycls = ycls.reindex(X.index)
    folds = []
    for s in range(TRAIN_MIN, len(X), TEST_LEN):  # calendar-anchored folds, partial last fold kept
        folds.append(bt_ml.Fold(X.index[: s - PURGE], X.index[s : s + TEST_LEN]))
    bt_ml.assert_folds_disjoint(folds, PURGE, X.index)
    return bt_ml.walk_forward_predict(X, ycls, model_factory(kind, c, depth), folds)


def build_signal(inputs, kind="logit", c=1.0, depth=2):
    pred = predict(inputs, kind, c, depth)
    sig = (pred > 0.5).astype(float).where(pred.notna())
    idx = inputs["px"].index
    return sig.reindex(sig.index.union(idx)).ffill().reindex(idx)


VARIANTS = {
    "logit_c0.1": dict(kind="logit", c=0.1),
    "logit_c1": dict(kind="logit", c=1.0),
    "gbm_d2": dict(kind="gbm", depth=2),
    "gbm_d3": dict(kind="gbm", depth=3),
}
PARAMS_TEXT = {n: f"{n}; 9 PIT features; yearly expanding refit, purge 1m; P>0.5; monthly; next_close; 5bp" for n in VARIANTS}

In [4]:
X, ycls, yex = features(inputs)
rows = {}
for name, p in {**VARIANTS, "naive_mean (baseline)": dict(kind="naive")}.items():
    pred = predict(inputs, **p)
    rows[name] = {**bt_ml.oos_summary(pred - 0.5, yex), "share P>0.5": float((pred.dropna() > 0.5).mean())}
print("Out-of-sample quality of P(up) - 0.5 against realised next-month excess return (all OOS months)")
display(pd.DataFrame(rows).T.round(3))
Xd = X.dropna()
lr = model_factory("logit", 1.0)().fit(Xd.iloc[:-1].to_numpy(), ycls.reindex(Xd.index).iloc[:-1].to_numpy())
coef = pd.Series(lr.est[-1].coef_[0], index=FEATURES).sort_values()
print("Logistic C=1 coefficients on standardised features, fitted on all months (plausibility check only):")
display(coef.round(3).to_frame("coef"))

Out-of-sample quality of P(up) - 0.5 against realised next-month excess return (all OOS months)


,n,ic,rank_ic,hit,share P>0.5
logit_c0.1,979.0,0.007,0.010,0.598,0.905
logit_c1,979.0,0.004,0.004,0.591,0.883
gbm_d2,979.0,0.014,0.022,0.570,0.792
gbm_d3,979.0,-0.000,-0.001,0.560,0.757
naive_mean (baseline),979.0,-0.017,-0.022,0.605,1.000


Logistic C=1 coefficients on standardised features, fitted on all months (plausibility check only):


,coef
tbill,-0.238
dp,-0.146
mom12,-0.019
ret1,0.019
credit,0.027
term,0.029
vol21,0.036
sma_dist,0.108
ep,0.228


## 3. Look-ahead checks

Every variant is recomputed with all inputs cut at past dates (monthly inputs: completed months only) and must equal the full-data signal on those dates (`stage2.truncation_test`). No backtest runs unless all pass.

In [5]:
N_CUTS = 6
trunc = {}
for name, params in VARIANTS.items():
    trunc[name] = s2.truncation_test(build_signal, inputs, n_cuts=N_CUTS, **params)
display(pd.DataFrame({k: v.row() for k, v in trunc.items()}).T)
bad = [k for k, v in trunc.items() if not v.passed]
assert not bad, f"look-ahead detected in variants {bad}: fix before any backtest"
print("all variants pass the truncation look-ahead test")

,cuts,mismatches,max_abs_diff,passed
logit_c0.1,6,0,0.0,True
logit_c1,6,0,0.0,True
gbm_d2,6,0,0.0,True
gbm_d3,6,0,0.0,True


all variants pass the truncation look-ahead test


## 4. Development-period sweep

Each variant is scored on the **dev** period only and registered as a trial. The best dev Sharpe is chosen; validation and holdout results are never used for the choice.

In [6]:
signals = {name: build_signal(inputs, **params) for name, params in VARIANTS.items()}
sweep, CHOSEN = s2.dev_sweep(px, cash, bench, signals, PARAMS_TEXT, trial_id=TRIAL_ID, strategy=STRATEGY,
                             notebook=NOTEBOOK, protocol=PROTOCOL, rebalance=REBALANCE)
display(sweep)
print("chosen on dev Sharpe only:", CHOSEN, "|", PARAMS_TEXT[CHOSEN])

,dev_sharpe,d_sma200,d_bh,time_in_market,switches_pa,dev_start
logit_c0.1,0.550299,-0.082006,0.015899,0.842932,0.821915,1946-03-12
logit_c1,0.445172,-0.187133,-0.089228,0.808866,0.913239,1946-03-12
gbm_d2,0.402266,-0.230038,-0.132134,0.701672,2.420084,1946-03-12
gbm_d3,0.441775,-0.19053,-0.092625,0.684502,2.671225,1946-03-12


chosen on dev Sharpe only: logit_c0.1 | logit_c0.1; 9 PIT features; yearly expanding refit, purge 1m; P>0.5; monthly; next_close; 5bp


## 5. Frozen evaluation of the chosen variant

Frozen primary metric (`PLAN.md` section 2): beat SMA200 and buy & hold in dev **and** val, one-sided block-bootstrap p <= 0.10 vs SMA200 in both, deflated Sharpe >= 0.95 with the registry trial count. Holdout is confirmatory only.

In [7]:
signal = signals[CHOSEN]
runs, cmp, stats, res = s2.evaluate(px, cash, bench, signal, protocol=PROTOCOL, rebalance=REBALANCE)
display(cmp.round(3))
display(stats)
display(res.to_frame().round(3))
print(f"n_trials={res.n_trials} | DSR={res.dsr:.3f} | passes={res.passes} | holdout confirms={res.holdout_confirms}")
display(pd.Series(res.checks, name="check"))

,strategy,bh,d_bh,sma200,d_sma200,mix60,d_mix60
dev,0.550,0.534,0.016,0.632,-0.082,0.541,0.009
val,0.271,0.319,-0.048,0.572,-0.301,0.309,-0.038
holdout,0.781,0.781,0.000,0.658,0.123,0.779,0.002
full,0.513,0.522,-0.009,0.621,-0.108,0.522,-0.009


,CAGR,Vol,Sharpe,Sortino,Max DD,Calmar,Worst day,MaxDD peak,MaxDD recovered,Final x,Time in market,Switches,Switches / yr,Cost drag / yr
dev,0.113158,0.123199,0.550299,0.752401,-0.340239,0.332584,-0.204547,1973-01-11,1976-01-08,109.444863,0.842932,36,0.821915,0.000411
val,0.07195,0.18045,0.271193,0.382333,-0.537043,0.133975,-0.090259,2007-10-09,-,4.011686,0.953797,6,0.300082,0.00015
holdout,0.142806,0.1715,0.780753,1.099612,-0.337905,0.422623,-0.119799,2020-02-19,2020-08-10,9.403162,1.0,0,0.0,0.0
full,0.108813,0.149889,0.512974,0.714764,-0.537043,0.202615,-0.204547,2007-10-09,2013-05-03,4128.537931,0.903231,42,0.521059,0.000261


,Sharpe,d_sma200,d_bh,p_vs_sma200
dev,0.550,-0.082,0.016,0.800
val,0.271,-0.301,-0.048,0.945
holdout,0.781,0.123,0.000,NaN
full,0.513,-0.108,-0.009,NaN


n_trials=27 | DSR=0.994 | passes=False | holdout confirms=True


dev_beats_sma200        False
dev_beats_bh             True
dev_p_le_0.1            False
val_beats_sma200        False
val_beats_bh            False
val_p_le_0.1            False
dsr_ge_0.95              True
holdout_d_sma200_pos     True
Name: check, dtype: bool

In [8]:
s2.plot_growth(runs, bench, f'{TRIAL_ID} {CHOSEN}', PROTOCOL).show()

## 6. Robustness: costs, placebo, peek test, decades

In [9]:
cs = bs.cost_sensitivity(px, signal, cash, rebalance=REBALANCE)
display(cs.round(3))
sma_full = bs.excess_sharpe(bench["sma200"].reindex(runs["full"].index), bench["tbill"].reindex(runs["full"].index))
be = bs.breakeven_cost_bps(px, signal, cash, bench_sharpe=sma_full, rebalance=REBALANCE)
print(f"break-even one-way cost vs SMA200 Sharpe over the same window ({sma_full:.3f}): {be:.1f} bp")
pl = bs.placebo_test(px, signal, cash, n_placebo=200, rebalance=REBALANCE, rng=np.random.default_rng(bs.SEED))
display(pd.Series(pl, name="placebo").to_frame())
peek = bt_pit.lookahead_test(px, signal, cash, rebalance=REBALANCE)
print(f"peek test: Sharpe by shift {peek.sharpe_by_shift.round(3).to_dict()} | gain {peek.peek_gain:+.3f}")
display(s2.decade_table(runs["full"], bench).style.format("{:.2f}"))

,Sharpe,CAGR,Max DD,turnover_pa
cost_bps,,,,
0,0.528,0.111,-0.537,0.528
5,0.526,0.111,-0.537,0.528
10,0.525,0.110,-0.537,0.528
25,0.519,0.109,-0.538,0.528
50,0.510,0.108,-0.539,0.528


break-even one-way cost vs SMA200 Sharpe over the same window (0.621): 0.0 bp


,placebo
stat,Sharpe
real,0.526391
placebo_mean,0.452332
placebo_q95,0.534144
quantile,0.925
p_one_sided,0.075
n_placebo,200
time_in_market,0.796867


peek test: Sharpe by shift {-21: 0.619, -5: 0.527, -1: 0.527, 0: 0.526, 1: 0.585, 5: 0.586, 21: 0.574} | gain +0.093


,Sharpe,Sharpe B&H,Sharpe SMA200,CAGR,CAGR B&H,CAGR SMA200,time in market
1940s,0.17,0.31,-0.25,0.02,0.05,-0.02,0.93
1950s,1.62,1.42,1.51,0.20,0.19,0.18,0.86
1960s,0.41,0.40,0.43,0.08,0.08,0.07,0.97
1970s,-0.02,0.03,0.37,0.05,0.06,0.09,0.83
1980s,0.54,0.53,0.58,0.16,0.17,0.17,0.68
1990s,0.94,0.92,0.64,0.18,0.18,0.13,0.92
2000s,-0.12,-0.05,0.50,-0.02,-0.01,0.08,0.99
2010s,0.90,0.90,0.71,0.14,0.14,0.09,1.00
2020s,0.67,0.67,0.58,0.15,0.15,0.10,1.00


## 7. Register the result

In [10]:
note = ("passes" if res.passes else "fails") + f" frozen metric; chosen {CHOSEN} on dev; protocol {PROTOCOL}"
rows = s2.register_final(TRIAL_ID, STRATEGY, NOTEBOOK, PARAMS_TEXT[CHOSEN], res, note=note, header=seed_registry.HEADER)
display(rows)
print("registry trial count:", br.trial_count())

,trial_id,date,strategy,notebook,params,period,sharpe,d_sma200,d_bh,p_vs_sma200,dsr,n_trials,passes,status,note
61,S08-v1/dev,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,logit_c0.1; 9 PIT features; yearly expanding r...,dev,0.5503,-0.0820,0.0159,,,,,run,"dev-only sweep, variant logit_c0.1"
62,S08-v2/dev,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,logit_c1; 9 PIT features; yearly expanding ref...,dev,0.4452,-0.1871,-0.0892,,,,,run,"dev-only sweep, variant logit_c1"
63,S08-v3/dev,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,gbm_d2; 9 PIT features; yearly expanding refit...,dev,0.4023,-0.2300,-0.1321,,,,,run,"dev-only sweep, variant gbm_d2"
64,S08-v4/dev,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,gbm_d3; 9 PIT features; yearly expanding refit...,dev,0.4418,-0.1905,-0.0926,,,,,run,"dev-only sweep, variant gbm_d3"
65,S08/dev,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,logit_c0.1; 9 PIT features; yearly expanding r...,dev,0.5503,-0.0820,0.0159,0.8001,,27,False,fail,fails frozen metric; chosen logit_c0.1 on dev;...
66,S08/val,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,logit_c0.1; 9 PIT features; yearly expanding r...,val,0.2712,-0.3010,-0.0479,0.9448,,27,False,fail,fails frozen metric; chosen logit_c0.1 on dev;...
67,S08/holdout,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,logit_c0.1; 9 PIT features; yearly expanding r...,holdout,0.7808,0.1232,0.0000,,,27,False,fail,fails frozen metric; chosen logit_c0.1 on dev;...
68,S08/full,2026-10-20,ml_walkforward,ml/ml_walkforward.ipynb,logit_c0.1; 9 PIT features; yearly expanding r...,full,0.5130,-0.1076,-0.0089,,0.9938,27,False,fail,fails frozen metric; chosen logit_c0.1 on dev;...


registry trial count: 27


## 8. Current signal

In [11]:
last = signal.dropna()
print(f"latest signal ({CHOSEN}) on {last.index[-1].date()}: equity weight {last.iloc[-1]:.2f}")
chk = bc.check_days(px.index, REBALANCE)
display(last[chk.reindex(last.index).fillna(False)].tail(6).to_frame("equity weight at decision"))
print("Research output only -- not a recommendation or an order.")

latest signal (logit_c0.1) on 2026-10-19: equity weight 1.00


,equity weight at decision
Date,
2026-04-30,1.0
2026-05-29,1.0
2026-06-30,1.0
2026-07-31,1.0
2026-08-31,1.0
2026-09-30,1.0


Research output only -- not a recommendation or an order.


## 9. Evidence summary (computed)

In [12]:
for line in s2.evidence_lines(res, trunc[CHOSEN], peek, pl, be):
    print("*", line)

* Truncation look-ahead test: PASSED on 6 cut dates; peek test gain +0.093.
* dev: Sharpe 0.550 (vs SMA200 -0.082, vs B&H +0.016, p vs SMA200 0.800)
* val: Sharpe 0.271 (vs SMA200 -0.301, vs B&H -0.048, p vs SMA200 0.945)
* holdout: Sharpe 0.781 (vs SMA200 +0.123, vs B&H +0.000)
* Deflated Sharpe 0.994 with n_trials=27.
* Placebo: real 0.526 vs placebo 95th pct 0.534 (p 0.075); break-even cost vs SMA200 full 0.0 bp.
* Verdict: FAILS the frozen primary metric (failed checks: dev_beats_sma200, dev_p_le_0.1, val_beats_sma200, val_beats_bh, val_p_le_0.1); holdout confirms.


## 10. Caveats

* Out-of-sample by construction (walk-forward, purge), but the *choice* of features and models is
  informed by the literature, which was written with hindsight over much of this sample.
* Monthly data: about 900 training months at most and ~11 positive / negative regimes per decade;
  machine learning has little data to learn from here.
* The OOS table above mixes dev, val and holdout months; it is descriptive only.
* Shiller valuation features are frozen after 2024-09.

## 11. Interpretation

* **No out-of-sample predictive skill.** Over ~980 walk-forward months the four models' rank correlation
  with the realised next-month excess return is -0.00 to +0.02 (naive training-mean baseline -0.02),
  and their hit rates (56-60%) do not beat the share of up-months (60.5%, the naive baseline).
* The best dev variant (logistic, C = 0.1) predicts "up" in ~90% of months, so it is close to buy & hold:
  dev Sharpe 0.55 vs SMA200 0.63 (p 0.80), val 0.27 vs 0.57 (p 0.94). **Fails** the frozen metric. In the
  holdout it was invested 100% of the time, i.e. exactly buy & hold (0.78).
* The gradient-boosting models switch more (2.4-2.7 times a year) and do worse in dev: more flexibility
  fits noise when there are only a few hundred monthly observations.
* The coefficient signs are economically plausible (high T-bill rate and dividend yield negative, high
  earnings yield and price above its SMA200 positive), so the model is not learning nonsense; the
  information is simply too weak at a one-month horizon. This matches Welch & Goyal's conclusion.